# SPY vs TLT Month-End Rebalancing Analysis
**Hypothesis:** Pension and insurance company rebalancing at month-end meaningfully impacts the relative performance of stocks vs bonds.

**Data:** SPY (S&P 500 ETF) and TLT (Long Duration Treasury ETF), June 2016 – June 2026

## Section 1: Load & Merge Data

In [1]:
import yfinance as yf

start_date = "2016-04-01"
end_date = "2026-04-01"

print(f"Fetching SPY and TLT daily closing prices from {start_date} to {end_date}...")

data = yf.download(["SPY", "TLT"], start=start_date, end=end_date, auto_adjust=True)

close = data["Close"][["SPY", "TLT"]].copy()
close.index.name = "Date"
close.to_csv("spy_tlt_closes.csv")
print(f"Saved {len(close)} rows to spy_tlt_closes.csv")

Fetching SPY and TLT daily closing prices from 2016-04-01 to 2026-04-01...


[*********************100%***********************]  2 of 2 completed


Saved 2514 rows to spy_tlt_closes.csv


In [2]:
import pandas as pd
from scipy import stats

# Load the combined CSV produced by yfinance
df = pd.read_csv('spy_tlt_closes.csv')

if df.columns[0] != 'Date':
    df = pd.read_csv('spy_tlt_closes.csv', header=[0,1])
    df.columns = ['Date', 'SPY', 'TLT']

# Parse Date and convert prices to numeric
df['Date'] = pd.to_datetime(df['Date'])
df['SPY']  = pd.to_numeric(df['SPY'], errors='coerce')
df['TLT']  = pd.to_numeric(df['TLT'], errors='coerce')

# Drop any rows with missing data, sort oldest to newest
df = df.dropna().sort_values('Date').reset_index(drop=True)

print(f'Total trading days: {len(df)}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
df.head()

Total trading days: 2514
Date range: 2016-04-01 to 2026-03-31


,Date,SPY,TLT
0,2016-04-01,175.345322,98.474556
1,2016-04-04,174.777542,98.534828
2,2016-04-05,173.031891,99.612488
3,2016-04-06,174.921631,98.889076
4,2016-04-07,172.828522,100.109810


## Section 2: Calculate Monthly Returns

For each month we identify 3 key dates:
- **d1** = first business day (start of month)
- **d2last** = second-to-last business day (end of main window)
- **dlast** = last business day (rebalancing day)

We then calculate:
- **main window return**: d1 → d2last (was it a strong month?)
- **final day return**: d2last → dlast (what happened on rebalancing day?)

In [3]:
results = []

for (year, month), group in df.groupby([df['Date'].dt.year, df['Date'].dt.month]):
    group = group.sort_values('Date')
    days = group['Date'].tolist()
    
    # Need at least 3 trading days to have d1, d2last, dlast
    if len(days) < 3:
        continue
    
    d1, d2last, dlast = days[0], days[-2], days[-1]
    
    # Pull prices for each key date
    spy1  = group.loc[group['Date']==d1,    'SPY'].values[0]
    spy2  = group.loc[group['Date']==d2last,'SPY'].values[0]
    spy3  = group.loc[group['Date']==dlast, 'SPY'].values[0]
    tlt1  = group.loc[group['Date']==d1,    'TLT'].values[0]
    tlt2  = group.loc[group['Date']==d2last,'TLT'].values[0]
    tlt3  = group.loc[group['Date']==dlast, 'TLT'].values[0]
    
    # Calculate returns: (end price / start price) - 1
    spy_main  = spy2/spy1 - 1
    tlt_main  = tlt2/tlt1 - 1
    spy_final = spy3/spy2 - 1
    tlt_final = tlt3/tlt2 - 1
    
    results.append({
        'Month':          f'{year}-{month:02d}',
        'First Bus Day':  d1.date(),
        '2nd-to-Last':    d2last.date(),
        'Last Bus Day':   dlast.date(),
        'SPY Main (%)':   round(spy_main*100, 4),
        'TLT Main (%)':   round(tlt_main*100, 4),
        'Main Spread (SPY-TLT %)': round((spy_main-tlt_main)*100, 4),
        'SPY Final (%)':  round(spy_final*100, 4),
        'TLT Final (%)':  round(tlt_final*100, 4),
        'Final Spread (SPY-TLT %)': round((spy_final-tlt_final)*100, 4),
        'spy_main_raw':   spy_main,
        'tlt_main_raw':   tlt_main,
        'spy_final_raw':  spy_final,
        'tlt_final_raw':  tlt_final,
        'Qualifies (SPY beat TLT by >5%)': (spy_main - tlt_main) > 0.05
    })

df_res = pd.DataFrame(results)
print(f'Complete months calculated: {len(df_res)}')
df_res.drop(columns=['spy_main_raw','tlt_main_raw','spy_final_raw','tlt_final_raw']).head(10)

Complete months calculated: 120


,Month,First Bus Day,2nd-to-Last,Last Bus Day,SPY Main (%),TLT Main (%),Main Spread (SPY-TLT %),SPY Final (%),TLT Final (%),Final Spread (SPY-TLT %),Qualifies (SPY beat TLT by >5%)
0,2016-04,2016-04-01,2016-04-28,2016-04-29,0.2561,-1.2473,1.5034,-0.5399,0.2557,-0.7956,False
1,2016-05,2016-05-02,2016-05-27,2016-05-31,1.0915,1.5882,-0.4967,-0.1902,0.2388,-0.4290,False
2,2016-06,2016-06-01,2016-06-29,2016-06-30,-1.2057,6.1110,-7.3168,1.3646,0.3685,0.9960,False
3,2016-07,2016-07-01,2016-07-28,2016-07-29,3.2631,-0.1281,3.3912,0.1614,0.8334,-0.6720,False
4,2016-08,2016-08-01,2016-08-30,2016-08-31,0.4886,-0.0715,0.5601,-0.2844,0.1432,-0.4276,False
5,2016-09,2016-09-01,2016-09-29,2016-09-30,-0.7478,-0.7298,-0.0180,0.7546,-0.8937,1.6483,False
6,2016-10,2016-10-03,2016-10-28,2016-10-31,-1.5015,-4.6488,3.1473,0.0047,0.6133,-0.6086,False
7,2016-11,2016-11-01,2016-11-29,2016-11-30,4.6917,-6.7094,11.4012,-0.2399,-1.6200,1.3801,True
8,2016-12,2016-12-01,2016-12-29,2016-12-30,2.7792,0.4476,2.3316,-0.3655,0.1513,-0.5169,False
9,2017-01,2017-01-03,2017-01-30,2017-01-31,1.0256,-0.3092,1.3348,-0.0088,0.6959,-0.7047,False


## Section 3: Filter Qualifying Months
Keep only months where SPY beat TLT by more than 5% in the main window.
These are the months where pension/insurance rebalancing pressure would be highest.

In [4]:
qualified = df_res[df_res['Qualifies (SPY beat TLT by >5%)']].copy()

print(f'Total complete months: {len(df_res)}')
print(f'Qualifying months (SPY beat TLT by >5%): {len(qualified)}')
print()

qualified[['Month','First Bus Day','2nd-to-Last','Last Bus Day',
           'SPY Main (%)','TLT Main (%)','Main Spread (SPY-TLT %)',
           'SPY Final (%)','TLT Final (%)','Final Spread (SPY-TLT %)']]


Total complete months: 120
Qualifying months (SPY beat TLT by >5%): 24



,Month,First Bus Day,2nd-to-Last,Last Bus Day,SPY Main (%),TLT Main (%),Main Spread (SPY-TLT %),SPY Final (%),TLT Final (%),Final Spread (SPY-TLT %)
7,2016-11,2016-11-01,2016-11-29,2016-11-30,4.6917,-6.7094,11.4012,-0.2399,-1.6200,1.3801
21,2018-01,2018-01-02,2018-01-30,2018-01-31,4.8331,-2.7731,7.6062,0.0497,0.5901,-0.5404
24,2018-04,2018-04-02,2018-04-27,2018-04-30,3.5305,-2.4532,5.9837,-0.7690,0.1767,-0.9457
33,2019-01,2019-01-02,2019-01-30,2019-01-31,6.9550,-0.9988,7.9537,0.8783,0.8600,0.0183
38,2019-06,2019-06-03,2019-06-27,2019-06-28,6.6823,0.3473,6.3350,0.5146,-0.0678,0.5824
41,2019-09,2019-09-03,2019-09-27,2019-09-30,2.0720,-3.0433,5.1153,0.4638,0.2452,0.2186
42,2019-10,2019-10-01,2019-10-30,2019-10-31,3.7171,-2.7223,6.4394,-0.2663,1.3489,-1.6153
48,2020-04,2020-04-01,2020-04-29,2020-04-30,19.1184,0.9997,18.1187,-0.9311,-1.1677,0.2366
49,2020-05,2020-05-01,2020-05-28,2020-05-29,7.1360,-3.2867,10.4227,0.4456,0.7142,-0.2686
52,2020-08,2020-08-03,2020-08-28,2020-08-31,6.6273,-5.1957,11.8230,-0.3622,0.6641,-1.0264


## Section 4: Statistical Analysis
Test whether TLT consistently beats SPY on the final rebalancing day.
A negative spread (SPY minus TLT) means TLT won — which is what the hypothesis predicts.

In [5]:
spread = qualified['spy_final_raw'] - qualified['tlt_final_raw']
t, p = stats.ttest_1samp(spread, 0)

stats_table = pd.DataFrame([
    ['Complete months tested',                      len(df_res)],
    ['Months where SPY beat TLT by >5% (2nd-to-last)', len(qualified)],
    ['Avg final-day SPY return',                    f"{qualified['spy_final_raw'].mean()*100:.4f}%"],
    ['Avg final-day TLT return',                    f"{qualified['tlt_final_raw'].mean()*100:.4f}%"],
    ['Avg final-day SPY minus TLT',                 f"{spread.mean()*100:.4f}%"],
    ['Median final-day SPY minus TLT',              f"{spread.median()*100:.4f}%"],
    ['SPY underperformed TLT',                      f"{(spread<0).sum()} / {len(spread)} months = {(spread<0).mean()*100:.1f}%"],
    ['t-stat on final-day spread',                  f"{t:.4f}"],
    ['p-value',                                     f"{p:.6f}"],
], columns=['Condition', 'Result'])

print('=== ANALYSIS 1: Month-End Rebalancing Effect ===')
stats_table

=== ANALYSIS 1: Month-End Rebalancing Effect ===


,Condition,Result
0,Complete months tested,120
1,Months where SPY beat TLT by >5% (2nd-to-last),24
2,Avg final-day SPY return,-0.3273%
3,Avg final-day TLT return,0.2584%
4,Avg final-day SPY minus TLT,-0.5857%
5,Median final-day SPY minus TLT,-0.2932%
6,SPY underperformed TLT,16 / 24 months = 66.7%
7,t-stat on final-day spread,-2.4500
8,p-value,0.022315


## Section 5: Trade P&L
For each qualifying month, simulate: **Long \$2M TLT / Short \$1M SPY** on the final rebalancing day.

P&L formula: `2,000,000 × TLT_1day_return − 1,000,000 × SPY_1day_return`

In [6]:
qualified = qualified.copy()
qualified['P&L ($)'] = (
    2_000_000 * qualified['tlt_final_raw'] - 
    1_000_000 * qualified['spy_final_raw']
).round(2)

# Full trade table
trade_table = qualified[['Month','2nd-to-Last','Last Bus Day',
                          'SPY Final (%)','TLT Final (%)','P&L ($)']].copy()
trade_table['Trade Window'] = trade_table['2nd-to-Last'].astype(str) + ' → ' + trade_table['Last Bus Day'].astype(str)
trade_table = trade_table[['Month','Trade Window','SPY Final (%)','TLT Final (%)','P&L ($)']]
trade_table['Win'] = trade_table['P&L ($)'] > 0

print('=== TRADE P&L TABLE ===')
print(trade_table.to_string(index=False))
print()

# Summary
pnl = qualified['P&L ($)']
summary = pd.DataFrame([
    ['Total P&L',           f"${pnl.sum():,.2f}"],
    ['Average P&L per trade',f"${pnl.mean():,.2f}"],
    ['Median P&L',          f"${pnl.median():,.2f}"],
    ['Winning trades',      f"{(pnl>0).sum()} / {len(pnl)} = {(pnl>0).mean()*100:.1f}%"],
    ['Losing trades',       f"{(pnl<0).sum()} / {len(pnl)} = {(pnl<0).mean()*100:.1f}%"],
    ['Best trade',          f"{qualified.loc[pnl.idxmax(),'Month']} (${pnl.max():,.2f})"],
    ['Worst trade',         f"{qualified.loc[pnl.idxmin(),'Month']} (${pnl.min():,.2f})"],
], columns=['Metric', 'Result'])

print('=== TRADE SUMMARY ===')
summary

=== TRADE P&L TABLE ===
  Month            Trade Window  SPY Final (%)  TLT Final (%)   P&L ($)   Win
2016-11 2016-11-29 → 2016-11-30        -0.2399        -1.6200 -30001.17 False
2018-01 2018-01-30 → 2018-01-31         0.0497         0.5901  11305.28  True
2018-04 2018-04-27 → 2018-04-30        -0.7690         0.1767  11223.41  True
2019-01 2019-01-30 → 2019-01-31         0.8783         0.8600   8417.39  True
2019-06 2019-06-27 → 2019-06-28         0.5146        -0.0678  -6501.05 False
2019-09 2019-09-27 → 2019-09-30         0.4638         0.2452    266.54  True
2019-10 2019-10-30 → 2019-10-31        -0.2663         1.3489  29641.73  True
2020-04 2020-04-29 → 2020-04-30        -0.9311        -1.1677 -14042.82 False
2020-05 2020-05-28 → 2020-05-29         0.4456         0.7142   9827.76  True
2020-08 2020-08-28 → 2020-08-31        -0.3622         0.6641  16905.15  True
2020-11 2020-11-27 → 2020-11-30        -0.4427        -0.1248   1930.59  True
2021-01 2021-01-28 → 2021-01-29        -

,Metric,Result
0,Total P&L,"$202,581.90"
1,Average P&L per trade,"$8,440.91"
2,Median P&L,"$8,202.08"
3,Winning trades,18 / 24 = 75.0%
4,Losing trades,6 / 24 = 25.0%
5,Best trade,"2021-02 ($71,270.08)"
6,Worst trade,"2016-11 ($-30,001.17)"


## Section 6: Next-Day Bounce Analysis
After a strong month + month-end selloff, does SPY bounce back the very next trading day?

**Filter:**
1. SPY beat TLT by ≥ 5% (main window)
2. SPY return on final day was negative (confirmed selloff)
3. Then measure: next trading day return (last bus day → first bus day of next month)

In [7]:
next_day_results = []

for _, row in qualified.iterrows():
    # Find the first trading day after the last business day of this month
    next_day_data = df[df['Date'] > pd.Timestamp(row['Last Bus Day'])].head(1)
    if len(next_day_data) == 0:
        continue
    
    next_day       = next_day_data['Date'].values[0]
    spy_next_price = next_day_data['SPY'].values[0]
    tlt_next_price = next_day_data['TLT'].values[0]
    spy_last_price = df.loc[df['Date']==pd.Timestamp(row['Last Bus Day']), 'SPY'].values[0]
    tlt_last_price = df.loc[df['Date']==pd.Timestamp(row['Last Bus Day']), 'TLT'].values[0]
    
    spy_next_ret = spy_next_price/spy_last_price - 1
    tlt_next_ret = tlt_next_price/tlt_last_price - 1
    
    next_day_results.append({
        'Month':              row['Month'],
        'Last Bus Day':       row['Last Bus Day'],
        'Next Trading Day':   pd.Timestamp(next_day).date(),
        'SPY Final Day (%)':  round(row['spy_final_raw']*100, 4),
        'SPY Next Day (%)':   round(spy_next_ret*100, 4),
        'TLT Next Day (%)':   round(tlt_next_ret*100, 4),
        'spy_final_raw':      row['spy_final_raw'],
        'spy_next_raw':       spy_next_ret,
    })

df_next = pd.DataFrame(next_day_results)

# Tighter filter: also had a negative final day (confirmed selloff happened)
df_bounce = df_next[df_next['spy_final_raw'] < 0].copy()

print('=== NEXT-DAY BOUNCE TABLE (strong month + negative final day) ===')
print(df_bounce[['Month','Last Bus Day','Next Trading Day',
                 'SPY Final Day (%)','SPY Next Day (%)','TLT Next Day (%)']].to_string(index=False))
print()

bounce_summary = pd.DataFrame([
    ['Number of cases',         len(df_bounce)],
    ['Avg next-day SPY return', f"{df_bounce['spy_next_raw'].mean()*100:.4f}%"],
    ['Median next-day SPY return', f"{df_bounce['spy_next_raw'].median()*100:.4f}%"],
    ['Positive next-day returns', f"{(df_bounce['spy_next_raw']>0).sum()} / {len(df_bounce)} ({(df_bounce['spy_next_raw']>0).mean()*100:.0f}%)"],
    ['Negative next-day returns', f"{(df_bounce['spy_next_raw']<0).sum()} / {len(df_bounce)} ({(df_bounce['spy_next_raw']<0).mean()*100:.0f}%)"],
], columns=['Metric', 'Value'])

print('=== NEXT-DAY BOUNCE SUMMARY ===')
bounce_summary

=== NEXT-DAY BOUNCE TABLE (strong month + negative final day) ===
  Month Last Bus Day Next Trading Day  SPY Final Day (%)  SPY Next Day (%)  TLT Next Day (%)
2016-11   2016-11-30       2016-12-01            -0.2399           -0.3675           -1.0558
2018-04   2018-04-30       2018-05-01            -0.7690            0.1777           -0.3602
2019-10   2019-10-31       2019-11-01            -0.2663            0.9264           -0.3078
2020-04   2020-04-30       2020-05-01            -0.9311           -2.6473            0.8563
2020-08   2020-08-31       2020-09-01            -0.3622            0.9419            1.1437
2020-11   2020-11-30       2020-12-01            -0.4427            1.0938           -1.4745
2021-01   2021-01-29       2021-02-01            -2.0020            1.6646            0.1159
2021-02   2021-02-26       2021-03-01            -0.5153            2.4240           -1.3207
2021-12   2021-12-31       2022-01-03            -0.2520            0.5790           -2.6250
2022

,Metric,Value
0,Number of cases,15
1,Avg next-day SPY return,0.4128%
2,Median next-day SPY return,0.5633%
3,Positive next-day returns,11 / 15 (73%)
4,Negative next-day returns,4 / 15 (27%)


## Section 7: Sharpe Ratio
The Sharpe ratio measures return earned per unit of risk.

**Formula:** `(Mean P&L / Std Dev) × √N`

- **Mean P&L** — average profit per trade
- **Std Dev** — how spread out the P&Ls are (risk)
- **√N** — scales by sample size

In [8]:
import numpy as np
pnl = qualified['P&L ($)']

n      = len(pnl)
mean   = pnl.mean()
std    = pnl.std(ddof=1)    # ddof=1 = sample std dev
sharpe = (mean / std) * np.sqrt(n)

sharpe_table = pd.DataFrame([
    ['N (number of trades)',   n],
    ['Sum of all P&Ls',        f'${pnl.sum():,.2f}'],
    ['Mean P&L',               f'${mean:,.2f}'],
    ['Variance',               f'${pnl.var(ddof=1):,.2f}'],
    ['Std Dev',                f'${std:,.2f}'],
    ['√N',                     f'{np.sqrt(n):.4f}'],
    ['Sharpe Ratio',           f'{sharpe:.4f}'],
], columns=['Metric', 'Value'])

print('=== SECTION 7: SHARPE RATIO ===')
sharpe_table

=== SECTION 7: SHARPE RATIO ===


,Metric,Value
0,N (number of trades),24
1,Sum of all P&Ls,"$202,581.90"
2,Mean P&L,"$8,440.91"
3,Variance,"$393,615,997.87"
4,Std Dev,"$19,839.76"
5,√N,4.8990
6,Sharpe Ratio,2.0843


A Sharpe of 2.08 shows the strategy generates strong risk-adjusted returns.
For every dollar of volatility taken on, the strategy earns roughly 2 dollars of
average profit. 